# SVM ABSA

#1. INSTALL AND IMPORT

In [7]:
!pip -q install iterative-stratification

In [8]:
from __future__ import annotations

import json
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    hamming_loss,
    precision_score,
    recall_score,
    make_scorer,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from sklearn.svm import LinearSVC

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

SEED = 33
np.random.seed(SEED)

## 2. Configuration

There is a total of 12 combinations:

- weighting: TF or TF-IDF;
- lowercasing: off or on;
- n-gram range: (1,1), (1,2), (1,3).


In [14]:
# ---- Data ----
DATA_PATH = "/content/annotation_test.json"

ASPECT_CATEGORIES = [
    "Baterija", "Kamera", "Ekran", "Memorija", "Zvučnici", "Izgled",
    "Hardver", "Softver", "Performanse", "Cena", "Opšta ocena"
]

# ---- Cross-validation ----
QUICK_MODE = False
OUTER_FOLDS = 3 if QUICK_MODE else 10
INNER_FOLDS = 2 if QUICK_MODE else 5
C_GRID = [0.1, 1.0] if QUICK_MODE else [0.01, 0.1, 1.0, 10.0]
N_JOBS = -1

# ---- Vectorization ----
MIN_DF = 1
MAX_DF = 1.0
MAX_FEATURES = None

# Primary metrics used to choose C inside nested CV.
ASPECT_TUNING_SCORER = "f1_macro"
POLARITY_TUNING_SCORER = "f1_macro"

OUTPUT_DIR = Path("/content/svm_absa_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load annotation data

In [15]:
def _parse_aspect_list(value):
    """Normalize an aspect annotation value to a list of dictionaries."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        value = json.loads(value)
    return value


def load_annotation_data(path: str) -> pd.DataFrame:
    df = pd.read_json(path)

    if "phone" not in df.columns:
        df["phone"] = ""

    df = df.copy()
    df["comment"] = df["comment"].fillna("").astype(str).str.strip()
    df["phone"] = df["phone"].fillna("").astype(str).str.strip()
    df["aspect_categories"] = df["aspect_categories"].map(_parse_aspect_list)

    df = df.reset_index(drop=True)
    df["review_id"] = np.arange(len(df), dtype=int)
    return df


df = load_annotation_data(DATA_PATH)
print("Reviews:", len(df))
df.head()

Reviews: 17551


,phone,comment,review_status,aspect_terms,aspect_categories,review_id
0,Huawei P50 Pro,Ćao ljudi. Čitala sam da Huawei P50 pro ima pr...,NE,[],[],0
1,Huawei P50 Pro,"Najkonkretnije me zanima, da li na huawei tele...",DA,"[{'fr': 386, 'to': 394, 'trg': 'kvalitet', 'ca...","[{'category': 'Izgled', 'polarity': 'Pozitivan...",1
2,Huawei P50 Pro,Huawei je brend kvalitet i sve napravljeno da ...,DA,"[{'fr': 16, 'to': 24, 'trg': 'kvalitet', 'cate...","[{'category': 'Izgled', 'polarity': 'Pozitivan'}]",2
3,Huawei P50 Pro,"Pozdrav svima, da li je neko uspeo da resi pro...",DA,"[{'fr': 54, 'to': 68, 'trg': 'notifikacijama',...","[{'category': 'Softver', 'polarity': 'Negativa...",3
4,Huawei P50 Pro,Da li neko zna kako da se proveri prva aktivac...,NE,[],[],4


In [16]:
def make_model_text(frame: pd.DataFrame) -> pd.Series:
    text = frame["comment"].fillna("").astype(str)
    return text


def clean_annotations_for_review(aspects, review_id=None):
    """Keep one polarity per category"""
    by_category = {}
    for item in aspects:
        if not isinstance(item, dict):
            continue
        category = item.get("category")
        polarity = item.get("polarity")
        if category not in ASPECT_CATEGORIES or polarity is None or str(polarity).strip() == "":
            continue
        polarity = str(polarity).strip()
        by_category[category] = polarity
    return by_category


def build_targets(frame: pd.DataFrame):
    aspect_sets = []
    polarity_rows = []
    model_texts = make_model_text(frame).reset_index(drop=True)

    for pos, (_, row) in enumerate(frame.iterrows()):
        by_category = clean_annotations_for_review(
            row["aspect_categories"], review_id=row["review_id"]
        )
        aspect_sets.append(list(by_category.keys()))
        model_text = model_texts.iloc[pos]
        for category, polarity in by_category.items():
            polarity_rows.append({
                "review_id": int(row["review_id"]),
                "text": model_text,
                "aspect": category,
                "polarity": polarity,
            })

    mlb = MultiLabelBinarizer(classes=ASPECT_CATEGORIES)
    y_aspect = mlb.fit_transform(aspect_sets)
    polarity_df = pd.DataFrame(polarity_rows)
    return y_aspect, polarity_df


X_text = make_model_text(df)
y_aspect, polarity_df = build_targets(df)

print("Aspect-polarity annotations:", len(polarity_df))

Aspect-polarity annotations: 14712


## 4. Dataset statistics


In [17]:
aspect_counts = pd.Series(y_aspect.sum(axis=0), index=ASPECT_CATEGORIES, name="count").sort_values(ascending=False)
polarity_counts = polarity_df["polarity"].value_counts()

stats = pd.DataFrame({
    "reviews": [len(df)],
    "aspect_annotations": [len(polarity_df)],
})
display(stats)
display(aspect_counts.to_frame())
display(polarity_counts.to_frame("count"))

,reviews,aspect_annotations
0,17551,14712


,count
Opšta ocena,3109
Baterija,2217
Softver,1924
Kamera,1448
Hardver,1412
Performanse,1196
Ekran,978
Cena,881
Izgled,862
Zvučnici,514


,count
polarity,
Pozitivan,7847
Negativan,5868
Konflikt,520
Neutralan,477


## 5. Preprocess config and model pipeline creation


In [18]:
@dataclass(frozen=True)
class PrepConfig:
    weighting: str       # "tf" or "tfidf"
    lowercase: bool
    max_ngram: int       # 1, 2, 3 means (1, max_ngram)

    @property
    def ngram_range(self):
        return (1, self.max_ngram)

    @property
    def name(self):
        lc = "lower-on" if self.lowercase else "lower-off"
        return f"{self.weighting}_{lc}_1-{self.max_ngram}gram"


def make_preprocessing_grid():
    return [
        PrepConfig(weighting=w, lowercase=lc, max_ngram=n)
        for w in ("tf", "tfidf")
        for lc in (False, True)
        for n in (1, 2)
    ]


PREPROCESSING_CONFIGS = make_preprocessing_grid()
if QUICK_MODE:
    PREPROCESSING_CONFIGS = PREPROCESSING_CONFIGS[:2]

pd.DataFrame([asdict(c) | {"name": c.name} for c in PREPROCESSING_CONFIGS])

,weighting,lowercase,max_ngram,name
0,tf,False,1,tf_lower-off_1-1gram
1,tf,False,2,tf_lower-off_1-2gram
2,tf,True,1,tf_lower-on_1-1gram
3,tf,True,2,tf_lower-on_1-2gram
4,tfidf,False,1,tfidf_lower-off_1-1gram
5,tfidf,False,2,tfidf_lower-off_1-2gram
6,tfidf,True,1,tfidf_lower-on_1-1gram
7,tfidf,True,2,tfidf_lower-on_1-2gram


In [19]:
def make_vectorizer(config: PrepConfig) -> TfidfVectorizer:
    return TfidfVectorizer(
        lowercase=config.lowercase,
        ngram_range=config.ngram_range,
        use_idf=(config.weighting == "tfidf"),
        smooth_idf=True,
        sublinear_tf=False,
        norm="l2",
        min_df=MIN_DF,
        max_df=MAX_DF,
        max_features=MAX_FEATURES,
    )


def build_aspect_pipeline(config: PrepConfig, C: float = 1.0) -> Pipeline:
    return Pipeline([
        ("vectorizer", make_vectorizer(config)),
        ("clf", OneVsRestClassifier(
            LinearSVC(C=C, max_iter=20000)
        )),
    ])


def build_polarity_pipeline(config: PrepConfig, C: float = 1.0) -> Pipeline:
    features = ColumnTransformer([
        ("text", make_vectorizer(config), "text"),
        ("aspect", OneHotEncoder(handle_unknown="ignore"), ["aspect"]),
    ])
    return Pipeline([
        ("features", features),
        ("clf", LinearSVC(C=C, max_iter=20000)),
    ])

## 5. Evaluation metric calculation


In [20]:
def multilabel_metrics(y_true, y_pred):
    return {
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }


def multiclass_metrics(y_true, y_pred):
    return {
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }


def summarize_cv(raw: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    rows = []
    keys = ["weighting", "lowercase", "max_ngram", "config_name"]
    for key_values, group in raw.groupby(keys, dropna=False):
        row = dict(zip(keys, key_values))
        for metric in metrics:
            row[f"{metric}_mean"] = group[metric].mean()
            row[f"{metric}_std"] = group[metric].std(ddof=1)
        modes = group["best_C"].mode()
        row["C_mode"] = modes.iloc[0] if len(modes) else np.nan
        row["mean_fit_seconds"] = group["fit_seconds"].mean()
        rows.append(row)
    return pd.DataFrame(rows).sort_values(f"{metrics[0]}_mean", ascending=False).reset_index(drop=True)


def safe_stratified_group_folds(y, groups, requested: int) -> int:
    tmp = pd.DataFrame({"y": np.asarray(y), "group": np.asarray(groups)})
    unique_groups_per_class = tmp.groupby("y")["group"].nunique()
    max_possible = int(unique_groups_per_class.min())
    n_splits = min(requested, max_possible)
    if n_splits < 2:
        raise ValueError(
            "Not enough distinct review groups in every polarity class for stratified grouped CV. "
            f"Minimum groups per class = {max_possible}."
        )
    if n_splits < requested:
        warnings.warn(
            f"Requested {requested} folds, but only {n_splits} are possible for this polarity split."
        )
    return n_splits

## 6. Cross validation — aspect category detection

Searching for the best preprocesor and hyperparameter values

In [21]:
def evaluate_aspect_config_nested_cv(
    X: pd.Series,
    y: np.ndarray,
    config: PrepConfig,
    outer_folds: int = OUTER_FOLDS,
    inner_folds: int = INNER_FOLDS,
    c_grid: Iterable[float] = C_GRID,
):
    outer = MultilabelStratifiedKFold(
        n_splits=outer_folds, shuffle=True, random_state=SEED
    )
    rows = []

    for fold, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        inner = MultilabelStratifiedKFold(
            n_splits=inner_folds, shuffle=True, random_state=SEED + fold
        )

        grid = GridSearchCV(
            estimator=build_aspect_pipeline(config),
            param_grid={"clf__estimator__C": list(c_grid)},
            scoring=ASPECT_TUNING_SCORER,
            cv=inner,
            n_jobs=N_JOBS,
            refit=True,
        )

        t0 = time.perf_counter()
        grid.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - t0
        y_pred = grid.predict(X_test)

        row = {
            "fold": fold,
            "weighting": config.weighting,
            "lowercase": config.lowercase,
            "max_ngram": config.max_ngram,
            "config_name": config.name,
            "best_C": grid.best_params_["clf__estimator__C"],
            "fit_seconds": fit_seconds,
        }
        row.update(multilabel_metrics(y_test, y_pred))
        rows.append(row)

    return pd.DataFrame(rows)


def run_aspect_preprocessing_experiments(X, y, configs=PREPROCESSING_CONFIGS):
    all_rows = []
    for i, config in enumerate(configs, start=1):
        print(f"[{i}/{len(configs)}] Aspect experiment: {config.name}")
        fold_results = evaluate_aspect_config_nested_cv(X, y, config)
        all_rows.append(fold_results)
        print(f"  macro-F1 = {fold_results['macro_f1'].mean():.4f}")

    raw = pd.concat(all_rows, ignore_index=True)
    summary = summarize_cv(
        raw,
        metrics=["macro_f1", "micro_f1", "macro_precision", "macro_recall", "weighted_f1", "subset_accuracy", "hamming_loss"],
    )
    return raw, summary

In [22]:
# Full run: 12 preprocessing variants × nested CV.
aspect_cv_raw, aspect_cv_summary = run_aspect_preprocessing_experiments(X_text, y_aspect)

aspect_cv_raw.to_csv(OUTPUT_DIR / "aspect_cv_folds.csv", index=False)
aspect_cv_summary.to_csv(OUTPUT_DIR / "aspect_preprocessing_summary.csv", index=False)

display(aspect_cv_summary)


[1/8] Aspect experiment: tf_lower-off_1-1gram
  macro-F1 = 0.5651
[2/8] Aspect experiment: tf_lower-off_1-2gram
  macro-F1 = 0.5767
[3/8] Aspect experiment: tf_lower-on_1-1gram
  macro-F1 = 0.5735
[4/8] Aspect experiment: tf_lower-on_1-2gram
  macro-F1 = 0.5863
[5/8] Aspect experiment: tfidf_lower-off_1-1gram
  macro-F1 = 0.5540
[6/8] Aspect experiment: tfidf_lower-off_1-2gram
  macro-F1 = 0.5708
[7/8] Aspect experiment: tfidf_lower-on_1-1gram
  macro-F1 = 0.5526
[8/8] Aspect experiment: tfidf_lower-on_1-2gram
  macro-F1 = 0.5840


,weighting,lowercase,max_ngram,config_name,macro_f1_mean,macro_f1_std,micro_f1_mean,micro_f1_std,macro_precision_mean,macro_precision_std,macro_recall_mean,macro_recall_std,weighted_f1_mean,weighted_f1_std,subset_accuracy_mean,subset_accuracy_std,hamming_loss_mean,hamming_loss_std,C_mode,mean_fit_seconds
0,tf,True,2,tf_lower-on_1-2gram,0.586254,0.007078,0.661933,0.010223,0.716826,0.028823,0.526630,0.009537,0.652164,0.010004,0.672611,0.008360,0.047410,0.001443,10.0,129.301946
1,tfidf,True,2,tfidf_lower-on_1-2gram,0.584036,0.010332,0.665427,0.008470,0.699909,0.039413,0.530027,0.011013,0.654898,0.008143,0.671813,0.007325,0.047456,0.001089,10.0,124.621088
2,tf,False,2,tf_lower-off_1-2gram,0.576746,0.009278,0.653652,0.010425,0.708435,0.040403,0.513948,0.011014,0.643063,0.009843,0.669249,0.009390,0.048233,0.001382,10.0,125.777610
3,tf,True,1,tf_lower-on_1-1gram,0.573548,0.018377,0.631870,0.012273,0.671981,0.024950,0.514167,0.018374,0.626466,0.012517,0.657855,0.005266,0.051445,0.001482,10.0,45.661151
4,tfidf,False,2,tfidf_lower-off_1-2gram,0.570844,0.009490,0.657470,0.009646,0.694843,0.043787,0.513645,0.009838,0.645171,0.009326,0.668680,0.007139,0.048140,0.001308,10.0,147.347475
5,tf,False,1,tf_lower-off_1-1gram,0.565060,0.017263,0.621251,0.015265,0.673238,0.030614,0.503250,0.017475,0.615868,0.015035,0.653011,0.009409,0.052812,0.001980,10.0,50.325020
6,tfidf,False,1,tfidf_lower-off_1-1gram,0.553967,0.018241,0.613696,0.014053,0.669896,0.033067,0.493211,0.019329,0.608225,0.013664,0.648054,0.009210,0.053890,0.001671,10.0,55.929870
7,tfidf,True,1,tfidf_lower-on_1-1gram,0.552605,0.014617,0.612940,0.011616,0.658608,0.025090,0.495994,0.015983,0.607706,0.011280,0.647029,0.007249,0.054231,0.001553,10.0,41.661357


## 7. Cross validation — polarity classification

Searching for the best preprocesor and hyperparameter values

In [23]:
def evaluate_polarity_config_nested_cv(
    polarity_data: pd.DataFrame,
    config: PrepConfig,
    outer_folds: int = OUTER_FOLDS,
    inner_folds: int = INNER_FOLDS,
    c_grid: Iterable[float] = C_GRID,
):
    X = polarity_data[["text", "aspect"]]
    y = polarity_data["polarity"].astype(str)
    groups = polarity_data["review_id"].to_numpy()

    actual_outer = safe_stratified_group_folds(y, groups, outer_folds)
    outer = StratifiedGroupKFold(
        n_splits=actual_outer, shuffle=True, random_state=SEED
    )

    rows = []
    for fold, (train_idx, test_idx) in enumerate(outer.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        g_train = groups[train_idx]

        actual_inner = safe_stratified_group_folds(y_train, g_train, inner_folds)
        inner = StratifiedGroupKFold(
            n_splits=actual_inner, shuffle=True, random_state=SEED + fold
        )

        grid = GridSearchCV(
            estimator=build_polarity_pipeline(config),
            param_grid={"clf__C": list(c_grid)},
            scoring=POLARITY_TUNING_SCORER,
            cv=inner,
            n_jobs=N_JOBS,
            refit=True,
        )

        t0 = time.perf_counter()
        grid.fit(X_train, y_train, groups=g_train)
        fit_seconds = time.perf_counter() - t0
        y_pred = grid.predict(X_test)

        row = {
            "fold": fold,
            "weighting": config.weighting,
            "lowercase": config.lowercase,
            "max_ngram": config.max_ngram,
            "config_name": config.name,
            "best_C": grid.best_params_["clf__C"],
            "fit_seconds": fit_seconds,
        }
        row.update(multiclass_metrics(y_test, y_pred))
        rows.append(row)

    return pd.DataFrame(rows)


def run_polarity_preprocessing_experiments(polarity_data, configs=PREPROCESSING_CONFIGS):
    all_rows = []
    for i, config in enumerate(configs, start=1):
        print(f"[{i}/{len(configs)}] Polarity experiment: {config.name}")
        fold_results = evaluate_polarity_config_nested_cv(polarity_data, config)
        all_rows.append(fold_results)
        print(f"  macro-F1 = {fold_results['macro_f1'].mean():.4f}")

    raw = pd.concat(all_rows, ignore_index=True)
    summary = summarize_cv(
        raw,
        metrics=["macro_f1", "macro_precision", "macro_recall", "weighted_f1", "accuracy"],
    )
    return raw, summary

In [25]:
polarity_cv_raw, polarity_cv_summary = run_polarity_preprocessing_experiments(polarity_df)

polarity_cv_raw.to_csv(OUTPUT_DIR / "polarity_cv_folds.csv", index=False)
polarity_cv_summary.to_csv(OUTPUT_DIR / "polarity_preprocessing_summary.csv", index=False)

display(polarity_cv_summary)

[1/8] Polarity experiment: tf_lower-off_1-1gram


KeyboardInterrupt: 

## 8. Select the best preprocessing variants


In [26]:
def config_from_summary_row(row) -> PrepConfig:
    return PrepConfig(
        weighting=str(row["weighting"]),
        lowercase=bool(row["lowercase"]),
        max_ngram=int(row["max_ngram"]),
    )


BEST_ASPECT_CONFIG = config_from_summary_row(aspect_cv_summary.iloc[0])
BEST_POLARITY_CONFIG = config_from_summary_row(polarity_cv_summary.iloc[0])

print("Best aspect preprocessing:", BEST_ASPECT_CONFIG)
print("Best polarity preprocessing:", BEST_POLARITY_CONFIG)

Best aspect preprocessing: PrepConfig(weighting='tf', lowercase=True, max_ngram=2)
Best polarity preprocessing: PrepConfig(weighting='tf', lowercase=True, max_ngram=2)


## 9. End-to-end Cross validation


In [9]:
def _fit_best_aspect_on_train(X_train, y_train, config: PrepConfig, fold_seed: int):
    inner = MultilabelStratifiedKFold(
        n_splits=INNER_FOLDS, shuffle=True, random_state=fold_seed
    )
    grid = GridSearchCV(
        build_aspect_pipeline(config),
        {"clf__estimator__C": C_GRID},
        scoring=ASPECT_TUNING_SCORER,
        cv=inner,
        n_jobs=N_JOBS,
        refit=True,
    )
    grid.fit(X_train, y_train)
    return grid.best_estimator_, grid.best_params_["clf__estimator__C"]


def _fit_best_polarity_on_train(pol_train: pd.DataFrame, config: PrepConfig, fold_seed: int):
    X_train = pol_train[["text", "aspect"]]
    y_train = pol_train["polarity"].astype(str)
    groups = pol_train["review_id"].to_numpy()
    actual_inner = safe_stratified_group_folds(y_train, groups, INNER_FOLDS)
    inner = StratifiedGroupKFold(
        n_splits=actual_inner, shuffle=True, random_state=fold_seed
    )
    grid = GridSearchCV(
        build_polarity_pipeline(config),
        {"clf__C": C_GRID},
        scoring=POLARITY_TUNING_SCORER,
        cv=inner,
        n_jobs=N_JOBS,
        refit=True,
    )
    grid.fit(X_train, y_train, groups=groups)
    return grid.best_estimator_, grid.best_params_["clf__C"]


def truth_pair_set(aspects):
    by_category = clean_annotations_for_review(aspects)
    return {f"{category}|||{polarity}" for category, polarity in by_category.items()}


def predict_pair_sets_for_reviews(review_frame, aspect_model, polarity_model):
    review_text = make_model_text(review_frame)
    aspect_pred = aspect_model.predict(review_text)
    predicted_sets = [set() for _ in range(len(review_frame))]

    pair_rows = []
    pair_positions = []
    for pos, row_pred in enumerate(aspect_pred):
        for j, present in enumerate(row_pred):
            if present:
                pair_rows.append({"text": review_text.iloc[pos], "aspect": ASPECT_CATEGORIES[j]})
                pair_positions.append(pos)

    if pair_rows:
        pair_frame = pd.DataFrame(pair_rows)
        polarity_pred = polarity_model.predict(pair_frame[["text", "aspect"]])
        for pos, row, polarity in zip(pair_positions, pair_rows, polarity_pred):
            predicted_sets[pos].add(f"{row['aspect']}|||{polarity}")

    return aspect_pred, predicted_sets


def evaluate_end_to_end_nested_cv(
    frame: pd.DataFrame,
    y_aspects: np.ndarray,
    polarity_data: pd.DataFrame,
    aspect_config: PrepConfig,
    polarity_config: PrepConfig,
):
    X = make_model_text(frame)
    outer = MultilabelStratifiedKFold(
        n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED
    )

    pair_classes = sorted({
        f"{row.aspect}|||{row.polarity}" for row in polarity_data.itertuples()
    })
    pair_mlb = MultiLabelBinarizer(classes=pair_classes)
    pair_mlb.fit([[]])

    oof_aspect_pred = np.zeros_like(y_aspects)
    oof_pair_pred = [set() for _ in range(len(frame))]
    fold_rows = []

    for fold, (train_idx, test_idx) in enumerate(outer.split(X, y_aspects), start=1):
        print(f"End-to-end fold {fold}/{OUTER_FOLDS}")
        train_ids = set(frame.iloc[train_idx]["review_id"].astype(int))

        aspect_model, aspect_C = _fit_best_aspect_on_train(
            X.iloc[train_idx], y_aspects[train_idx], aspect_config, SEED + 100 + fold
        )

        pol_train = polarity_data[polarity_data["review_id"].isin(train_ids)].copy()
        polarity_model, polarity_C = _fit_best_polarity_on_train(
            pol_train, polarity_config, SEED + 200 + fold
        )

        test_frame = frame.iloc[test_idx].copy()
        aspect_pred, pair_pred_sets = predict_pair_sets_for_reviews(
            test_frame, aspect_model, polarity_model
        )
        oof_aspect_pred[test_idx] = aspect_pred
        for global_idx, pair_set in zip(test_idx, pair_pred_sets):
            oof_pair_pred[global_idx] = pair_set

        true_sets = [truth_pair_set(v) for v in test_frame["aspect_categories"]]
        y_pair_true = pair_mlb.transform(true_sets)
        y_pair_pred = pair_mlb.transform(pair_pred_sets)

        row = {
            "fold": fold,
            "aspect_C": aspect_C,
            "polarity_C": polarity_C,
        }
        for k, v in multilabel_metrics(y_aspects[test_idx], aspect_pred).items():
            row[f"aspect_{k}"] = v
        pair_metrics = multilabel_metrics(y_pair_true, y_pair_pred)
        for k, v in pair_metrics.items():
            row[f"pair_{k}"] = v
        fold_rows.append(row)

    fold_df = pd.DataFrame(fold_rows)
    true_pair_sets = [truth_pair_set(v) for v in frame["aspect_categories"]]
    y_pair_true_all = pair_mlb.transform(true_pair_sets)
    y_pair_pred_all = pair_mlb.transform(oof_pair_pred)

    return {
        "folds": fold_df,
        "oof_aspect_pred": oof_aspect_pred,
        "pair_classes": pair_classes,
        "oof_pair_pred_sets": oof_pair_pred,
        "y_pair_true": y_pair_true_all,
        "y_pair_pred": y_pair_pred_all,
    }

In [1]:
end_to_end = evaluate_end_to_end_nested_cv(
    df,
    y_aspect,
    polarity_df,
    BEST_ASPECT_CONFIG,
    BEST_POLARITY_CONFIG,
)

end_to_end["folds"].to_csv(OUTPUT_DIR / "end_to_end_cv_folds.csv", index=False)
display(end_to_end["folds"])

print("\nMean end-to-end aspect-polarity pair metrics:")
pair_columns = [c for c in end_to_end["folds"].columns if c.startswith("pair_")]
display(end_to_end["folds"][pair_columns].agg(["mean", "std"]).T)

print("\nPer-aspect out-of-fold classification report:")
print(classification_report(
    y_aspect,
    end_to_end["oof_aspect_pred"],
    target_names=ASPECT_CATEGORIES,
    zero_division=0,
))

NameError: name 'evaluate_end_to_end_nested_cv' is not defined

In [33]:
end_to_end["folds"]

,fold,aspect_C,polarity_C,aspect_macro_precision,aspect_macro_recall,aspect_macro_f1,aspect_micro_f1,aspect_weighted_f1,aspect_subset_accuracy,aspect_hamming_loss,pair_macro_precision,pair_macro_recall,pair_macro_f1,pair_micro_f1,pair_weighted_f1,pair_subset_accuracy,pair_hamming_loss
0,1,10.0,10.0,0.696493,0.532869,0.589642,0.673993,0.663633,0.678632,0.046102,0.272591,0.210587,0.229208,0.558242,0.529021,0.659829,0.015618
1,2,10.0,10.0,0.719158,0.517188,0.587880,0.647390,0.636974,0.665527,0.048640,0.273936,0.203090,0.224446,0.513706,0.482557,0.643875,0.016770
2,3,10.0,10.0,0.672149,0.539874,0.581919,0.667385,0.657332,0.669516,0.047967,0.243619,0.209413,0.218629,0.517960,0.487889,0.646724,0.017379
3,4,10.0,10.0,0.764424,0.535857,0.599724,0.674341,0.663602,0.681481,0.045429,0.324692,0.213258,0.235621,0.533977,0.502332,0.656980,0.016252
4,5,10.0,10.0,0.715899,0.525043,0.588958,0.657748,0.649314,0.664387,0.047708,0.246368,0.196861,0.213468,0.511334,0.483542,0.643305,0.017029
5,6,10.0,10.0,0.689759,0.531163,0.586983,0.655870,0.646151,0.664387,0.048433,0.275787,0.203403,0.219589,0.501288,0.468570,0.640456,0.017547
6,7,10.0,10.0,0.744758,0.514612,0.576315,0.653789,0.644640,0.664387,0.048278,0.411857,0.218845,0.250953,0.513373,0.485548,0.645584,0.016965
7,8,10.0,10.0,0.696587,0.517602,0.575127,0.649598,0.639881,0.678815,0.049596,0.265583,0.202613,0.217008,0.512070,0.482213,0.650911,0.017265
8,9,10.0,10.0,0.723405,0.516549,0.587921,0.667424,0.658550,0.671795,0.045480,0.270932,0.194525,0.215301,0.521212,0.488508,0.646154,0.016369
9,10,10.0,10.0,0.745631,0.535540,0.588069,0.671789,0.661567,0.687179,0.046465,0.274029,0.208757,0.223122,0.534211,0.500557,0.659259,0.016485


## 10. Train final models on all available data

Nested CV above is used for unbiased evaluation. For deployment, the selected preprocessing configurations are retained and `C` is tuned once more on the complete annotated dataset before fitting the final estimators.

In [29]:
def fit_final_models(frame, y_aspects, polarity_data, aspect_config, polarity_config):
    X = make_model_text(frame)

    aspect_cv = MultilabelStratifiedKFold(
        n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED + 500
    )
    aspect_grid = GridSearchCV(
        build_aspect_pipeline(aspect_config),
        {"clf__estimator__C": C_GRID},
        scoring=ASPECT_TUNING_SCORER,
        cv=aspect_cv,
        n_jobs=N_JOBS,
        refit=True,
    )
    aspect_grid.fit(X, y_aspects)

    pol_X = polarity_data[["text", "aspect"]]
    pol_y = polarity_data["polarity"].astype(str)
    pol_groups = polarity_data["review_id"].to_numpy()
    pol_folds = safe_stratified_group_folds(pol_y, pol_groups, OUTER_FOLDS)
    polarity_cv = StratifiedGroupKFold(
        n_splits=pol_folds, shuffle=True, random_state=SEED + 600
    )
    polarity_grid = GridSearchCV(
        build_polarity_pipeline(polarity_config),
        {"clf__C": C_GRID},
        scoring=POLARITY_TUNING_SCORER,
        cv=polarity_cv,
        n_jobs=N_JOBS,
        refit=True,
    )
    polarity_grid.fit(pol_X, pol_y, groups=pol_groups)

    return {
        "aspect_model": aspect_grid.best_estimator_,
        "polarity_model": polarity_grid.best_estimator_,
        "aspect_C": aspect_grid.best_params_["clf__estimator__C"],
        "polarity_C": polarity_grid.best_params_["clf__C"],
    }


final_models = fit_final_models(
    df,
    y_aspect,
    polarity_df,
    BEST_ASPECT_CONFIG,
    BEST_POLARITY_CONFIG,
)
print("Final aspect C:", final_models["aspect_C"])
print("Final polarity C:", final_models["polarity_C"])

Final aspect C: 10.0
Final polarity C: 10.0


In [34]:
final_models

{'aspect_model': Pipeline(steps=[('vectorizer',
                  TfidfVectorizer(ngram_range=(1, 2), use_idf=False)),
                 ('clf',
                  OneVsRestClassifier(estimator=LinearSVC(C=10.0,
                                                          max_iter=20000)))]),
 'polarity_model': Pipeline(steps=[('features',
                  ColumnTransformer(transformers=[('text',
                                                   TfidfVectorizer(ngram_range=(1,
                                                                                2),
                                                                   use_idf=False),
                                                   'text'),
                                                  ('aspect',
                                                   OneHotEncoder(handle_unknown='ignore'),
                                                   ['aspect'])])),
                 ('clf', LinearSVC(C=10.0, max_iter=20000))]),
 'aspect_C':

## 11. Prediction Function

In [ ]:
def predict_absa(
    comments: str | list[str],
    phones: str | list[str] | None = None,
    models=final_models,
) -> pd.DataFrame:
    if isinstance(comments, str):
        comments = [comments]
    comments = list(comments)

    if phones is None:
        phones = [""] * len(comments)
    elif isinstance(phones, str):
        phones = [phones] * len(comments)
    else:
        phones = list(phones)

    if len(phones) != len(comments):
        raise ValueError("phones and comments must have the same length")

    input_df = pd.DataFrame({"phone": phones, "comment": comments})
    input_df["review_id"] = np.arange(len(input_df))

    aspect_pred, pair_sets = predict_pair_sets_for_reviews(
        input_df,
        models["aspect_model"],
        models["polarity_model"],
    )

    rows = []
    for i, pair_set in enumerate(pair_sets):
        for pair in sorted(pair_set):
            aspect, polarity = pair.split("|||", 1)
            rows.append({
                "input_index": i,
                "aspect": aspect,
                "polarity": polarity,
            })
    return pd.DataFrame(rows, columns=["input_index", "aspect", "polarity"])


# Example:
predict_absa("Baterija traje odlično, ali je kamera veoma loša.")

,input_index,aspect,polarity
0,0,Baterija,Pozitivan
1,0,Kamera,Pozitivan


## 12. Save models and all experiment tables

The saved object contains both trained models, the selected preprocessing settings, the aspect list, and the chosen `C` values.

In [30]:
artifact = {
    "aspect_model": final_models["aspect_model"],
    "polarity_model": final_models["polarity_model"],
    "aspect_categories": ASPECT_CATEGORIES,
    "best_aspect_config": asdict(BEST_ASPECT_CONFIG),
    "best_polarity_config": asdict(BEST_POLARITY_CONFIG),
    "aspect_C": final_models["aspect_C"],
    "polarity_C": final_models["polarity_C"]
}

model_path = OUTPUT_DIR / "svm_absa_models.joblib"
joblib.dump(artifact, model_path)
print("Saved:", model_path)
print("Results directory:", OUTPUT_DIR)

Saved: /content/svm_absa_results/svm_absa_models.joblib
Results directory: /content/svm_absa_results
